In [1]:
!pip install tiktoken

In [2]:
import psutil # for ram
import torch
import os
import random
import numpy as np
import torch.nn as nn
import tiktoken as tkm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

#  tiktoken/GPT-2
tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=10000,                  # <- the main knob
    min_frequency=2,                   # ignore ultra-rare merges
    special_tokens=["<|endoftext|>"],
)
tok.train(["serbian-training-data.txt"], trainer)
tok.save("misovic-bpe.json")

with open("serbian-training-data.txt", "r", encoding="utf-8") as f:
    data = f.read()
token_ids = tok.encode(data).ids # convert to manual char level tokenization - token_ids, where data is the training corpus

In [4]:
#INPUTS, tokenization and embedding
vocab_size = tok.get_vocab_size()
embed_dim = 256
batch_size = 20
block_size = 64 # how big is the context window for X and Y, where Y is X shifted to the right, aka the next element after X
num_heads = 4
num_layers = 6 # how many blocks of attention/ff

In [5]:
#using the architechture to construct the lm

class lm(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, block_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim) # return the token embed_dim sized vector
        self.pos_emb = nn.Embedding(block_size, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            batch_first=True
        )
        self.blocks = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False) # return logits for the whole vocab
        self.lm_head.weight = self.embedding.weight  #weight tying

    def forward(self, batch):
        B,T = batch.shape
        pos = torch.arange(T,device=batch.device)
        x = self.embedding(batch) + self.pos_emb(pos)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=batch.device)
        x = self.blocks(x, mask=mask, is_causal=True) # namesti encoder da bude kao decoder
        logits = self.lm_head(x)

        return logits

model = lm(vocab_size, embed_dim, num_heads, num_layers, block_size)
model = model.to(device)

In [6]:
#load the previous model
if os.path.exists("model_checkpoint.pt"):
    model.load_state_dict(torch.load("model_checkpoint.pt", map_location=device))
    print("Resumed from checkpoint.")

In [7]:
#training the model - making the dataset class

class LanguageModelDataset(torch.utils.data.Dataset):
    def __init__(self, token_ids, block_size):
        self.token_ids = token_ids
        self.block_size = block_size

    def __len__(self):
        return len(self.token_ids) - self.block_size

    def __getitem__(self, idx):
        x = torch.tensor(self.token_ids[idx : idx + self.block_size], dtype=torch.long)
        y = torch.tensor(self.token_ids[idx + 1 : idx + self.block_size + 1], dtype=torch.long)
        return x, y

from torch.utils.data import DataLoader

N = len(token_ids)
chunk=10000
chunks = [token_ids[i:i+chunk] for i in range(0,N,chunk)]
random.seed(42)
random.shuffle(chunks)
token_ids = [t for ch in chunks for t in ch]
train_ids = token_ids[:int(0.8 * N)]
val_ids   = token_ids[int(0.8 * N):int(0.9 * N)]
test_ids  = token_ids[int(0.9 * N):]

#training data is for train_ids
dataset = LanguageModelDataset(train_ids, block_size)
dataset_val = LanguageModelDataset(val_ids, block_size)
dataloader = DataLoader(dataset, batch_size = batch_size, shuffle=True, drop_last=True)
val_dataloader = DataLoader(dataset_val, batch_size = batch_size, shuffle=True, drop_last=True)

import torch.optim as optim

# Adam optimizer & epoch count:
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
num_epochs = 2


loss_fn = nn.CrossEntropyLoss()
try:
    for epoch in range(num_epochs):
        c = 0
        total_loss = 0
        if epoch >= 1:
            for param_group in optimizer.param_groups:
                param_group['lr'] *= 0.33
            print("Learning rate lowered!")
        for batch_X, batch_Y in dataloader:
            #if c >= 25000: break

            #move from the cpu to gpu
            batch_X = batch_X.to(device)
            batch_Y = batch_Y.to(device)
            #forward pass
            logits = model(batch_X) # should be batch_size, block_size, vocab_size
            #print(logits.shape)
            # compute loss
            loss = loss_fn(logits.view(-1, vocab_size), batch_Y.view(-1))
            optimizer.zero_grad()  # clear old gradients
            loss.backward()        # backprop / compute gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()       # update weights
            total_loss += loss.item()
            c+=1
            if c % 100 == 0 and psutil.virtual_memory().percent > 90:
                print("RAM is gettin filled up too much, stopping and saving...")
                torch.save(model.state_dict(), "model_checkpoint.pt")
                raise KeyboardInterrupt
            if c % 500 == 0:
                print(f"Iteration: {c}, loss: {loss.item():.4f}")
            if c % 5000 == 0:
                model.eval()
                with torch.no_grad():
                    vX, vY = next(iter(val_dataloader))
                    v_logits = model(vX.to(device))
                    v_loss = loss_fn(v_logits.view(-1, vocab_size), vY.to(device).view(-1))
                model.train()
                print(f"iter {c}: train {loss.item():.3f} | val {v_loss.item():.3f}")
        torch.save(model.state_dict(), "model_checkpoint.pt")
except KeyboardInterrupt:
    print("Training interrupted. Saving model...")
    torch.save(model.state_dict(), "model_checkpoint.pt")
    print("Model saved.")

Iteration: 500, loss: 18.5422
Iteration: 1000, loss: 14.3284
Iteration: 1500, loss: 12.3237
Iteration: 2000, loss: 10.5769
Iteration: 2500, loss: 9.4469
Iteration: 3000, loss: 8.8432
Iteration: 3500, loss: 8.8499
Iteration: 4000, loss: 8.0388
Iteration: 4500, loss: 7.6508
Iteration: 5000, loss: 7.7813
iter 5000: train 7.781 | val 7.407
Iteration: 5500, loss: 7.5971
Iteration: 6000, loss: 7.4665
Iteration: 6500, loss: 7.2112
Iteration: 7000, loss: 7.0915
Iteration: 7500, loss: 6.9453
Iteration: 8000, loss: 6.9454
Iteration: 8500, loss: 7.0432
Iteration: 9000, loss: 6.8729
Iteration: 9500, loss: 7.0599
Iteration: 10000, loss: 6.7309
iter 10000: train 6.731 | val 6.873
Iteration: 10500, loss: 6.8101
Iteration: 11000, loss: 6.5234
Iteration: 11500, loss: 6.6779
Iteration: 12000, loss: 6.4589
Iteration: 12500, loss: 6.5881
Iteration: 13000, loss: 6.4245
Iteration: 13500, loss: 6.3392
Iteration: 14000, loss: 6.3754
Iteration: 14500, loss: 6.4005
Iteration: 15000, loss: 6.0664
iter 15000: tra

In [8]:
#outputing the next token
prompt = input("Enter your prompt: ")
#logits = model(encoded_prompt)   →   (1, T, 10000)
#lm_head = (10000, 256)
def generate(prompt, max_tokens, temperature=0.7):
    model.eval() # no dropout anymore
    ids = tok.encode(prompt).ids
    encoded_prompt = torch.tensor([ids], dtype=torch.long, device=device) # [] to make it 2D
    with torch.no_grad():
        for _ in range(max_tokens):
            context = encoded_prompt[:, -block_size:] #model limitted to block_size, take the last block_size amount of tokens
            logits = model(context) #id's of the prompt, this gives us the next token
            last = logits[:, -1, :] # token id is the index of the matrix itself
            #[dim0:dim1:dim2], dim0 - batch, dim1 - token id out of them all, dim2- vocab
            probs = torch.softmax(last / temperature, dim=-1) #-1, ditch the other dimension, last is shape(1,10000) no T since we -1'd it in logits
            next_token = torch.multinomial(probs, num_samples=1) #multinomial picks 40% chance 40% of the time

            encoded_prompt = torch.cat([encoded_prompt, next_token], dim=1)
        decoded_prompt = tok.decode(encoded_prompt[0].tolist()) #back to text
    return decoded_prompt

generate(prompt, 200) #run the model

Enter your prompt: И тако кренувши


'И тако кренувши се, да се умерава и да не учини што је дотле. Кад би се то био пратилац у своме месту, она се удаљаваше од узбуђења.\nНа ове речи Заврзан рече:\n— Да л’ би било што је Станко знао да је то каква земља. Драшковић дође у мрак, све је доста.\n— Шта је ово, то је? — рече он.\n— Не треба, рече.\n— Па шта ћеш? — упита га Станко.\n— Тако је.\n— Шта?\n— Дох!...\nПа, на то се он и не одговори.\n— Ништа.\n— Не знам, не знам.\n— Баш сам то.\n— Па грађанство! — рече Станко.\n— Али у мени је! — рече Маринко.\n— Ако хоћеш, зашт’о, онда ћемо се опет лакше побити — рече Станко.\n— Шта ти је! А ти ћеш све управо. Ту ти сад то'

In [9]:
#print parameters of model & tokens of dataset
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Dataset tokens: {len(dataset):,}")

Model parameters: 7,314,944
Dataset tokens: 2,902,368
